## Carga del dataset y división Train/Test

En esta etapa se carga el dataset preprocesado y se separan las variables predictoras (`X`) de la variable objetivo (`y`).

Posteriormente, se realiza una división estratificada de los datos:
- 80% para entrenamiento.
- 20% para prueba.

La estratificación permite mantener la proporción original de clases en ambos conjuntos de datos.

In [4]:
import os
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
TEST_SIZE = 0.2
TARGET_COL = "is_recommended"

# Ruta del dataset (con ../ para salir de la carpeta notebooks)
DATA_PATH = "../data/processed/sephora_limpio.csv"

# Cargar dataset
df = pd.read_csv(DATA_PATH)

# Se toma una muestra aleatoria para acelerar GridSearchCV
df = df.sample(50000, random_state=RANDOM_STATE)

# Validación de columna objetivo
if TARGET_COL not in df.columns:
    raise ValueError(f"No se encontró la columna objetivo '{TARGET_COL}' en el dataset")

# Eliminar NaN en la variable objetivo (por seguridad)
df = df.dropna(subset=[TARGET_COL]).copy()

# Separar X e y
y = df[TARGET_COL]
X = df.drop(columns=[TARGET_COL])

# Split estratificado
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Shape X_train:", X_train.shape)
print("Shape X_test:", X_test.shape)
print("Proporción y_train:", y_train.value_counts(normalize=True).to_dict())
print("Proporción y_test:", y_test.value_counts(normalize=True).to_dict())

Shape X_train: (33706, 43)
Shape X_test: (8427, 43)
Proporción y_train: {1.0: 0.8411558772918768, 0.0: 0.15884412270812318}
Proporción y_test: {1.0: 0.8411059689094577, 0.0: 0.15889403109054231}


## Optimización de hiperparámetros con GridSearchCV

En esta etapa se utiliza `GridSearchCV` para buscar automáticamente la mejor combinación de hiperparámetros para el modelo `GradientBoostingClassifier`.

Se evaluarán distintas configuraciones del modelo utilizando validación cruzada (`cv=3`) y la métrica F1-score como criterio principal de evaluación.

El objetivo es mejorar el rendimiento del modelo y obtener una mejor capacidad de clasificación.

In [5]:
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV

# Modelo base
gb = GradientBoostingClassifier(random_state=RANDOM_STATE)

# Malla (grid) realista y acotada
param_grid = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.05, 0.1, 0.2],
    "max_depth": [2, 3],  
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 3],
    "subsample": [0.8, 1.0],
}

grid = GridSearchCV(
    estimator=gb,
    param_grid=param_grid,
    scoring="f1",
    cv=3,
    n_jobs=-1,
    verbose=1,
    refit=True,
)

print("Tamaño del grid:", len(param_grid["n_estimators"]) * len(param_grid["learning_rate"]) * len(param_grid["max_depth"]) * len(param_grid["min_samples_split"]) * len(param_grid["min_samples_leaf"]) * len(param_grid["subsample"]))

Tamaño del grid: 144


## Resultados de la optimización

Una vez ejecutado GridSearchCV, se obtienen:

- Los mejores hiperparámetros encontrados.
- El mejor F1-score promedio obtenido durante la validación cruzada.

Estos resultados permiten identificar la configuración más eficiente del modelo.

In [6]:
grid.fit(X_train, y_train)

print("\nMejores parámetros (best_params_):")
print(grid.best_params_)

print("\nMejor F1-score en entrenamiento (CV):")
print(f"{grid.best_score_:.6f}")

Fitting 3 folds for each of 144 candidates, totalling 432 fits

Mejores parámetros (best_params_):
{'learning_rate': 0.05, 'max_depth': 2, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200, 'subsample': 1.0}

Mejor F1-score en entrenamiento (CV):
0.978358


## Guardado del modelo entrenado

Finalmente, el mejor modelo encontrado es almacenado utilizando `joblib`.

Esto permite reutilizar posteriormente el modelo entrenado sin necesidad de volver a ejecutar el proceso completo de entrenamiento y optimización.

In [7]:
# Ruta corregida con ../
BEST_MODEL_PATH = "../models/trained_models/mejor_gradient_boosting.pkl"

# Asegurar que exista el directorio
os.makedirs(os.path.dirname(BEST_MODEL_PATH), exist_ok=True)

# El mejor estimador ya está entrenado con refit=True
best_model = grid.best_estimator_

joblib.dump(best_model, BEST_MODEL_PATH)

print(f"Modelo guardado exitosamente en: {BEST_MODEL_PATH}")

Modelo guardado exitosamente en: ../models/trained_models/mejor_gradient_boosting.pkl
